In [0]:
# =============================================================================
# Smart ERP – Stock Silver Layer Simulation  (Databricks / Delta Lake)
# Catalog  : smart_odoo
# Targets  :
#     silver.stock_warehouse              (10 Egypt warehouses — once only)
#     silver.stock_location               (3–6 locations per warehouse per run)
#     silver.stock_lot                    (50 lots per run)
#     silver.stock_picking                (240 per run: receipt/delivery/internal/return)
#     silver.stock_move                   (1–4 moves per picking)
#     silver.stock_move_line              (1 per move)
#     silver.stock_quant                  (net position per product × location)
#     silver.stock_warehouse_orderpoint   (1 per warehouse per run)
#     silver.stock_scrap                  (20 per run)
#
# ════════════════════════════════════════════════════════════════════════════
# CONNECTIONS TO OTHER SILVER TABLES  ← KEY FIX vs old script
# ════════════════════════════════════════════════════════════════════════════
#
#   product_id / product_name
#     ← silver.product_template  (product_tmpl_id, name, list_price)
#     ← silver.product_product   (product_id, standard_price / cost)
#
#   partner_id (vendors, id 1–50)
#     ← silver.res_partner  WHERE customer_rank = 0 (suppliers)
#     Used for: Receipts, Internal transfers
#
#   partner_id (customers, id 51–250)
#     ← silver.res_partner  WHERE customer_rank > 0
#     Used for: Deliveries, Returns
#
#   Pickings origin field
#     Receipts   → 'PO-{purchase_id}'   (vendor replenishment)
#     Deliveries → 'SO-{sale_order_id}' references silver.sale_order
#
#   price_unit in stock_move
#     Receipts   → standard_price  (cost, what we paid the vendor)
#     Deliveries → list_price      (sale price, what customer pays)
#     Internal   → standard_price
#
# ════════════════════════════════════════════════════════════════════════════
# STOCK FLOW LOGIC
# ════════════════════════════════════════════════════════════════════════════
#
#   RECEIPTS  (picking_type=1, is_in=True)
#     → 150 per run  (more than deliveries → stock always positive)
#     → state: done 60% | assigned 25% | confirmed 10% | cancel 5%
#     → adds quantity to stock_quant when state=done
#
#   DELIVERIES  (picking_type=2, is_out=True)
#     → 120 per run
#     → state: done 50% | assigned 30% | confirmed 15% | cancel 5%
#     → removes quantity from stock_quant when state=done
#     → reserved_quantity in quant when state=assigned
#
#   INTERNAL  (picking_type=3)
#     → 30 per run  (warehouse-to-warehouse transfers)
#     → no partner, no price impact
#
#   RETURNS  (picking_type=4)
#     → 10 per run  (reverse of a delivery)
#     → same customer partner as original delivery
#
# ════════════════════════════════════════════════════════════════════════════
# IDEMPOTENCY / SAFETY
# ════════════════════════════════════════════════════════════════════════════
#   ✅ All IDs: MAX(existing) + 1 — safe on every re-run
#   ✅ Warehouses: written only once (skip if code already exists)
#   ✅ stock_quant: per-product net computed from this run's done moves
#   ✅ Quantities always positive (receipts > deliveries by design)
# =============================================================================

import random
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
    BooleanType, TimestampType,
)

# ── Spark setup ───────────────────────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_stock_delta").getOrCreate()

CATALOG    = "smart_odoo"
SCHEMA     = "silver"
SOURCE_DB  = "python_sim"
COMPANY_ID = 1
COMPANY    = "Smart Odoo LLC"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 6, 15)

now        = datetime.now(timezone.utc).replace(tzinfo=None)

def rand_date() -> datetime:
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def maybe_date(prob_none: float = 0.3):
    return None if random.random() < prob_none else rand_date()

# ── Load reference data from silver ──────────────────────────────────────────
# Products: id → (name, list_price, standard_price)
products_map = {}
for r in spark.sql("""
    SELECT t.product_tmpl_id AS pid,
           t.name,
           t.list_price,
           p.standard_price
    FROM   silver.product_template t
    JOIN   silver.product_product  p ON p.product_id = t.product_tmpl_id
""").collect():
    products_map[r.pid] = {
        "name":           r.name,
        "list_price":     float(r.list_price),
        "standard_price": float(r.standard_price),
    }

assert len(products_map) > 0, "❌ No products found in silver.product_template"
pid_list = sorted(products_map.keys())   # [1 … 100]

# Vendors: partner_id 1–50 (customer_rank = 0)
vendors_map = {}
for r in spark.sql("""
    SELECT partner_id, partner_name
    FROM   silver.res_partner
    WHERE  customer_rank = 0
    ORDER BY partner_id
""").collect():
    vendors_map[r.partner_id] = r.partner_name

# Customers: partner_id 51–250 (customer_rank > 0)
customers_map = {}
for r in spark.sql("""
    SELECT partner_id, partner_name
    FROM   silver.res_partner
    WHERE  customer_rank > 0
    ORDER BY partner_id
""").collect():
    customers_map[r.partner_id] = r.partner_name

# Sale orders for linking delivery origins
sale_order_ids = []
try:
    sale_order_ids = [
        r.sale_order_id for r in spark.sql("""
            SELECT sale_order_id
            FROM   silver.sale_order
            WHERE  order_state IN ('sale', 'done')
            LIMIT  500
        """).collect()
    ]
except Exception:
    sale_order_ids = list(range(1, 501))   # fallback if table empty

vendor_list   = sorted(vendors_map.keys())
customer_list = sorted(customers_map.keys())

print(f"  Products loaded   : {len(products_map)}")
print(f"  Vendors loaded    : {len(vendors_map)}")
print(f"  Customers loaded  : {len(customers_map)}")
print(f"  Sale orders avail : {len(sale_order_ids)}")

# ── UOMs & routes ─────────────────────────────────────────────────────────────
UOMS   = {1: "Unit(s)", 2: "kg", 3: "liters", 4: "pcs", 5: "boxes"}
ROUTES = {1: "Buy", 2: "Manufacture", 3: "Resupply"}

# ── Warehouse definitions ─────────────────────────────────────────────────────
WH_CITIES = [
    ("Cairo WH1",   "CAI-01"),
    ("Cairo WH2",   "CAI-02"),
    ("Cairo WH3",   "CAI-03"),
    ("Alex WH1",    "ALX-01"),
    ("Alex WH2",    "ALX-02"),
    ("Giza WH",     "GZ-01"),
    ("Delta WH",    "DLT-01"),
    ("Mansoura WH", "MNS-01"),
    ("Tanta WH",    "TNT-01"),
    ("Aswan WH",    "ASN-01"),
]
RECEPTION_STEPS = ["one_step", "two_steps", "three_steps"]
DELIVERY_STEPS  = ["one_step", "two_steps", "three_steps"]

PICKING_TYPES = {
    1: "Receipts",
    2: "Delivery Orders",
    3: "Internal Transfers",
    4: "Returns",
}

# ── MAX ID helpers ────────────────────────────────────────────────────────────
def max_id(table: str, col: str) -> int:
    try:
        return int(spark.sql(
            f"SELECT COALESCE(MAX({col}), 0) FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()[0][0])
    except Exception:
        return 0

wh_id_start    = max_id("stock_warehouse",            "warehouse_id")                  + 1
loc_id_start   = max_id("stock_location",             "stock_location_id")             + 1
lot_id_start   = max_id("stock_lot",                  "stock_lot_id")                  + 1
pick_id_start  = max_id("stock_picking",              "stock_picking_id")              + 1
move_id_start  = max_id("stock_move",                 "stock_move_id")                 + 1
ml_id_start    = max_id("stock_move_line",            "stock_move_line_id")            + 1
quant_id_start = max_id("stock_quant",                "stock_quant_id")                + 1
op_id_start    = max_id("stock_warehouse_orderpoint", "stock_warehouse_orderpoint_id") + 1
scrap_id_start = max_id("stock_scrap",                "stock_scrap_id")                + 1

print(f"\n  warehouse start     : {wh_id_start}")
print(f"  location start      : {loc_id_start}")
print(f"  lot start           : {lot_id_start}")
print(f"  picking start       : {pick_id_start}")
print(f"  move start          : {move_id_start}")
print(f"  move_line start     : {ml_id_start}")
print(f"  quant start         : {quant_id_start}")
print(f"  orderpoint start    : {op_id_start}")
print(f"  scrap start         : {scrap_id_start}")

# ═════════════════════════════════════════════════════════════════════════════
# 1. WAREHOUSES  (10 fixed Egypt locations — written once, skipped on re-run)
# ═════════════════════════════════════════════════════════════════════════════
try:
    existing_codes = {
        r.code for r in
        spark.sql(f"SELECT code FROM {CATALOG}.{SCHEMA}.stock_warehouse").collect()
    }
except Exception:
    existing_codes = set()

wh_rows = []
wh_id   = wh_id_start

for wh_name, code in WH_CITIES:
    if code in existing_codes:
        continue
    vendor_id = random.choice(vendor_list)
    wh_rows.append(Row(
        warehouse_id    = wh_id,
        company_id      = COMPANY_ID,
        company_name    = COMPANY,
        partner_id      = vendor_id,
        partner_name    = vendors_map[vendor_id],
        warehouse_name  = wh_name,
        code            = code,
        reception_steps = random.choice(RECEPTION_STEPS),
        delivery_steps  = random.choice(DELIVERY_STEPS),
        active          = True,
        created_at      = now,
        updated_at      = now,
        dwh_loaded_at   = now,
        dwh_source_db   = SOURCE_DB,
    ))
    wh_id += 1

# Load full warehouse list (including pre-existing) for FK usage
try:
    all_wh = [
        (r.warehouse_id, r.warehouse_name) for r in
        spark.sql(
            f"SELECT warehouse_id, warehouse_name FROM {CATALOG}.{SCHEMA}.stock_warehouse"
        ).collect()
    ]
except Exception:
    all_wh = []

for r in wh_rows:
    all_wh.append((r.warehouse_id, r.warehouse_name))

if not all_wh:
    all_wh = [(i, f"WH-{i}") for i in range(wh_id_start, wh_id_start + 10)]

all_wh_ids = [wid for wid, _ in all_wh]
wh_name_map = {wid: wname for wid, wname in all_wh}

# ═════════════════════════════════════════════════════════════════════════════
# 2. LOCATIONS  (3–6 per warehouse per run)
# ═════════════════════════════════════════════════════════════════════════════
loc_rows  = []
loc_id    = loc_id_start
wh_to_locs: dict[int, list[int]] = {wh: [] for wh in all_wh_ids}

USAGE_POOL = (["internal"] * 85) + (["transit"] * 10) + (["view"] * 5)

for wh in all_wh_ids:
    for j in range(random.randint(3, 6)):
        loc_name = f"LOC-{wh}-{j}"
        barcode  = f"LOC-{random.randint(100000,999999)}" if random.random() > 0.2 else None
        inv_date = maybe_date(0.4)
        next_inv = (inv_date + timedelta(days=random.randint(30, 180))) if inv_date else None

        loc_rows.append(Row(
            stock_location_id   = loc_id,
            company_id          = COMPANY_ID,
            company_name        = COMPANY,
            location_id         = loc_id,
            location_name       = loc_name,
            warehouse_id        = wh,
            stock_location_name = loc_name,
            complete_name       = f"{wh_name_map.get(wh, f'WH{wh}')}/{loc_name}",
            usage               = random.choice(USAGE_POOL),
            parent_path         = f"/{wh}/{loc_id}/",
            barcode             = barcode,
            active              = True,
            replenish_location  = random.random() < 0.3,
            last_inventory_date = inv_date,
            next_inventory_date = next_inv,
            created_at          = now,
            updated_at          = now,
            dwh_loaded_at       = now,
            dwh_source_db       = SOURCE_DB,
        ))
        wh_to_locs[wh].append(loc_id)
        loc_id += 1

# Also load existing locations so quant/move FKs are valid
try:
    for r in spark.sql(
        f"SELECT stock_location_id, warehouse_id FROM {CATALOG}.{SCHEMA}.stock_location"
    ).collect():
        wh_to_locs.setdefault(r.warehouse_id, []).append(r.stock_location_id)
except Exception:
    pass

all_loc_ids = [lid for lids in wh_to_locs.values() for lid in lids]

# ═════════════════════════════════════════════════════════════════════════════
# 3. LOTS  (50 per run — linked to real product_ids)
# ═════════════════════════════════════════════════════════════════════════════
lot_rows    = []
lot_id      = lot_id_start
lot_id_pool = []

for _ in range(50):
    pid      = random.choice(pid_list)
    prod     = products_map[pid]
    loc_ref  = random.choice(all_loc_ids) if all_loc_ids else lot_id

    lot_rows.append(Row(
        stock_lot_id   = lot_id,
        product_id     = pid,
        product_name   = prod["name"],          # ← real product name
        company_id     = COMPANY_ID,
        company_name   = COMPANY,
        location_id    = loc_ref,
        location_name  = f"LOC-{loc_ref}",
        lot_name       = f"LOT-{now.year}-{lot_id:04d}",
        ref            = f"REF-{random.randint(1000, 9999)}",
        note           = random.choice([None, None, "Check before use", "Priority batch"]),
        lot_properties = None,
        standard_price = str(round(prod["standard_price"], 2)),  # ← real cost
        avg_cost       = round(prod["standard_price"] * random.uniform(0.95, 1.05), 2),
        created_at     = now,
        updated_at     = now,
        dwh_loaded_at  = now,
        dwh_source_db  = SOURCE_DB,
    ))
    lot_id_pool.append(lot_id)
    lot_id += 1

# ═════════════════════════════════════════════════════════════════════════════
# 4 + 5 + 6.  PICKINGS → MOVES → MOVE LINES
#
# Distribution per run:
#   150 Receipts   (type=1)  IN  from vendor    → adds stock
#   120 Deliveries (type=2)  OUT to customer    → removes stock
#    30 Internal   (type=3)  WH transfers       → neutral
#    10 Returns    (type=4)  customer returns   → adds stock back
#
# Price logic:
#   Receipts/Internal/Returns → price_unit = standard_price (cost)
#   Deliveries                → price_unit = list_price (sale price)
# ═════════════════════════════════════════════════════════════════════════════

pick_rows = []
move_rows = []
ml_rows   = []

pick_id   = pick_id_start
move_id   = move_id_start
ml_id     = ml_id_start

# Existing pickings for backorder refs
try:
    prior_pick_ids = [
        r.stock_picking_id for r in
        spark.sql(
            f"SELECT stock_picking_id FROM {CATALOG}.{SCHEMA}.stock_picking LIMIT 500"
        ).collect()
    ]
except Exception:
    prior_pick_ids = []

new_pick_ids = []   # accumulates within this run for cross-referencing

# ── State pools by picking type ───────────────────────────────────────────────
STATE_POOLS = {
    1: (["done"]*60 + ["assigned"]*25 + ["confirmed"]*10 + ["cancel"]*5),
    2: (["done"]*50 + ["assigned"]*30 + ["confirmed"]*15 + ["cancel"]*5),
    3: (["done"]*40 + ["assigned"]*40 + ["draft"]*20),
    4: (["done"]*70 + ["confirmed"]*30),
}

PRIORITIES = (["0"]*70 + ["1"]*20 + ["2"]*10)
PROC_METHODS = (["make_to_stock"]*80 + ["make_to_order"]*20)

# ── Accumulate ON-HAND qty per (product, location) from done moves ────────────
#
# WHY a simple receipt-only accumulator instead of receipt - delivery:
#
#   Receipts credit dest_loc.  Deliveries debit src_loc.
#   Because src/dest are chosen independently, a receipt for (pid, locA) and a
#   delivery for (pid, locB) land in DIFFERENT dict keys.  Subtracting them
#   produces near-zero "on-hand" on receipt rows and negative quants on
#   delivery rows — both wrong.
#
#   The correct model:
#     1. Credit receipts to dest_loc (adds stock to that bin).
#     2. Debit DONE deliveries from src_loc (removes stock from that bin).
#     3. After the loop, reserved_qty = on_hand × random(0.15–0.45)
#        because "assigned" deliveries reserve from whatever is already there,
#        regardless of which specific picking created the stock.
#
# Structure: {(product_id, dest_loc): qty_received_done}
receipt_quant: dict[tuple, float] = {}   # only receipts (is_in, done)
delivery_quant: dict[tuple, float] = {}  # only deliveries (is_out, done)

def _add_receipt(pid, lid, qty: float):
    k = (pid, lid)
    receipt_quant[k] = receipt_quant.get(k, 0.0) + qty

def _add_delivery(pid, lid, qty: float):
    k = (pid, lid)
    delivery_quant[k] = delivery_quant.get(k, 0.0) + qty

# ── Build each picking type ───────────────────────────────────────────────────
PICKING_PLAN = (
    [(1, 150)] +   # Receipts
    [(2, 120)] +   # Deliveries
    [(3, 30)]  +   # Internal
    [(4, 10)]      # Returns
)

for pt_id, count in PICKING_PLAN:
    for _ in range(count):

        wh       = random.choice(all_wh_ids)
        wh_locs  = wh_to_locs.get(wh, all_loc_ids) or all_loc_ids
        if len(wh_locs) < 2:
            wh_locs = all_loc_ids

        src_loc  = random.choice(wh_locs)
        dest_loc = random.choice([l for l in wh_locs if l != src_loc] or wh_locs)

        state      = random.choice(STATE_POOLS[pt_id])
        pick_date  = rand_date()
        sched_date = pick_date + timedelta(days=random.randint(0, 5))
        deadline   = sched_date + timedelta(days=random.randint(1, 14))
        date_done  = (pick_date + timedelta(days=random.randint(1, 3))) if state == "done" else None

        # ── Partner logic by picking type ────────────────────────────────────
        if pt_id == 1:    # Receipt: vendor supplies us
            partner_id   = random.choice(vendor_list)
            partner_name = vendors_map[partner_id]
        elif pt_id in (2, 4):   # Delivery / Return: customer
            partner_id   = random.choice(customer_list)
            partner_name = customers_map[partner_id]
        else:             # Internal: no external partner
            partner_id   = random.choice(vendor_list)
            partner_name = vendors_map[partner_id]

        # ── Origin references ─────────────────────────────────────────────────
        if pt_id == 1:
            origin = f"PO-{random.randint(1000, 9999)}"
        elif pt_id == 2 and sale_order_ids:
            origin = f"SO-{random.choice(sale_order_ids)}"
        elif pt_id == 4 and new_pick_ids:
            origin = f"Return of PICK-{random.choice(new_pick_ids):06d}"
        else:
            origin = None

        # ── Backorder reference ───────────────────────────────────────────────
        backorder_pool = prior_pick_ids + new_pick_ids
        backorder_id   = (
            random.choice(backorder_pool)
            if backorder_pool and random.random() < 0.15
            else None
        )

        user_id   = random.randint(1, 10)
        pick_name = f"PICK-{pick_id:06d}"
        is_in     = (pt_id in (1, 4))
        is_out    = (pt_id == 2)

        pick_rows.append(Row(
            stock_picking_id   = pick_id,
            company_id         = COMPANY_ID,
            company_name       = COMPANY,
            partner_id         = partner_id,
            partner_name       = partner_name,
            picking_type_id    = pt_id,
            picking_type_name  = PICKING_TYPES[pt_id],
            location_id        = src_loc,
            location_name      = f"LOC-{src_loc}",
            location_dest_id   = dest_loc,
            location_dest_name = f"LOC-{dest_loc}",
            user_id            = user_id,
            user_name          = f"User_{user_id}",
            backorder_id       = backorder_id,
            return_id          = str(backorder_id) if backorder_id and random.random() < 0.1 else None,
            stock_picking_name = pick_name,
            origin             = origin,
            move_type          = random.choice(["direct", "one", "two_steps"]),
            state              = state,
            priority           = random.choice(PRIORITIES),
            note               = random.choice([None, None, "Fragile", "Urgent", "Cold chain"]),
            shipping_weight    = round(random.uniform(0.5, 50.0), 2),
            has_deadline_issue = state in ("waiting", "confirmed") and random.random() < 0.2,
            printed            = state in ("assigned", "done"),
            is_locked          = state == "done",
            scheduled_date     = sched_date,
            date_deadline      = deadline,
            date_done          = date_done,
            created_at         = pick_date,
            updated_at         = pick_date,
            dwh_loaded_at      = now,
            dwh_source_db      = SOURCE_DB,
        ))
        new_pick_ids.append(pick_id)

        # ── MOVES (exactly 2 per picking) ────────────────────────────────────
        for _ in range(2):
            pid      = random.choice(pid_list)
            prod     = products_map[pid]
            uom_id   = 1    # Unit(s) — primary UOM for all products
            # product_uom_qty (ordered): always 10–50
            # quantity (actually moved): 10–50 when done, 0 otherwise
            product_uom_qty = float(random.randint(10, 50))
            qty             = product_uom_qty if state == "done" else 0.0

            # Price: cost for IN moves, list price for OUT moves
            price_unit = (
                prod["list_price"]     if pt_id == 2
                else prod["standard_price"]
            )
            value    = round(price_unit * product_uom_qty, 2)
            picked   = (state == "done")
            proc     = random.choice(PROC_METHODS)

            loc_final = random.choice(wh_locs) if random.random() > 0.5 else None
            ori_ret   = (
                random.choice(new_pick_ids)
                if new_pick_ids and random.random() < 0.05
                else None
            )

            move_rows.append(Row(
                stock_move_id             = move_id,
                company_id               = COMPANY_ID,
                company_name             = COMPANY,
                product_id               = pid,
                product_name             = prod["name"],       # ← real name
                product_uom_id           = uom_id,
                product_uom_name         = UOMS[uom_id],
                location_id              = src_loc,
                location_name            = f"LOC-{src_loc}",
                location_dest_id         = dest_loc,
                location_dest_name       = f"LOC-{dest_loc}",
                location_final_id        = loc_final,
                location_final_name      = f"LOC-{loc_final}" if loc_final else None,
                partner_id               = partner_id if random.random() > 0.3 else None,
                partner_name             = partner_name if random.random() > 0.3 else None,
                picking_id               = pick_id,
                picking_name             = pick_name,
                picking_type_id          = pt_id,
                picking_type_name        = PICKING_TYPES[pt_id],
                warehouse_id             = wh,
                warehouse_name           = wh_name_map.get(wh, f"WH-{wh}"),
                origin_returned_move_id  = ori_ret,
                origin_returned_move_name= f"PICK-{ori_ret:06d}" if ori_ret else None,
                priority                 = random.choice(PRIORITIES),
                state                    = state,
                origin                   = origin,
                reference                = pick_name,
                procure_method           = proc,
                inventory_name           = None,
                next_serial              = None,
                next_serial_count        = None,
                product_qty              = product_uom_qty,        # ordered qty
                product_uom_qty          = product_uom_qty,        # ordered qty
                quantity                 = qty,                    # moved qty (0 if not done)
                price_unit               = price_unit,
                value                    = round(price_unit * product_uom_qty, 2),  # based on ordered
                picked                   = picked,
                propagate_cancel         = state == "cancel",
                is_inventory             = False,
                additional               = False,
                to_refund                = random.random() < 0.05,
                is_in                    = is_in,
                is_out                   = is_out,
                is_dropship              = random.random() < 0.05,
                date                     = pick_date,
                date_deadline            = (
                    pick_date + timedelta(days=random.randint(1, 10))
                    if random.random() > 0.3 else None
                ),
                reservation_date         = (
                    pick_date - timedelta(days=random.randint(0, 3))
                    if state in ("assigned", "done") else None
                ),
                created_at               = pick_date,
                updated_at               = pick_date,
                dwh_loaded_at            = now,
                dwh_source_db            = SOURCE_DB,
            ))

            # ── Accumulate quant impact ───────────────────────────────────────
            if state == "done":
                if is_in:
                    _add_receipt(pid, dest_loc, float(qty))   # stock arrives at dest
                elif is_out:
                    _add_delivery(pid, src_loc, float(qty))   # stock leaves from src

            # ── MOVE LINE (1 per move) ────────────────────────────────────────
            lot_ref = (
                random.choice(lot_id_pool)
                if lot_id_pool and random.random() > 0.5
                else None
            )
            ml_rows.append(Row(
                stock_move_line_id     = ml_id,
                move_id                = move_id,
                picking_id             = str(pick_id),
                company_id             = COMPANY_ID,
                company_name           = COMPANY,
                product_id             = pid,
                product_name           = prod["name"],   # ← real name
                product_uom_id         = uom_id,
                product_uom_name       = UOMS[uom_id],
                location_id            = src_loc,
                location_name          = f"LOC-{src_loc}",
                location_dest_id       = dest_loc,
                location_dest_name     = f"LOC-{dest_loc}",
                lot_id                 = lot_ref,
                lot_name               = f"LOT-{now.year}-{lot_ref:04d}" if lot_ref else None,
                state                  = state,
                quantity               = qty,                       # moved (0 if not done)
                quantity_product_uom   = product_uom_qty,          # ordered
                picked                 = picked,
                is_entire_pack         = random.random() < 0.2,
                date                   = pick_date,
                created_at             = pick_date,
                updated_at             = pick_date,
                dwh_loaded_at          = now,
                dwh_source_db          = SOURCE_DB,
            ))

            move_id += 1
            ml_id   += 1

        pick_id += 1

# ═════════════════════════════════════════════════════════════════════════════
# 7. STOCK QUANT
#    Computed from this run's done moves — quantity is always >= 0.
#    For each (product, location) pair:
#      quantity         = receipt_done - delivery_done   (≥ 0 enforced)
#      reserved_qty     = min(delivery_assigned, quantity)
#      inventory_qty    = quantity ± small cycle-count drift
# ═════════════════════════════════════════════════════════════════════════════
quant_rows = []
quant_id   = quant_id_start

# Build the net on-hand map: receipt_quant - delivery_quant per (pid, loc)
# All location keys that appear in either dict
all_quant_keys = set(receipt_quant.keys()) | set(delivery_quant.keys())

for (pid, lid) in all_quant_keys:
    prod       = products_map[pid]
    received   = receipt_quant.get((pid, lid), 0.0)
    delivered  = delivery_quant.get((pid, lid), 0.0)

    qty = max(0.0, round(received - delivered, 0))
    if qty == 0:
        continue   # nothing on-hand at this bin — skip

    # reserved_quantity: fraction of on-hand that is committed to open deliveries.
    # Using a random 15–45% range gives a realistic warehouse picture where
    # some stock is freely available and some is already promised to customers.
    reserved = round(qty * random.uniform(0.15, 0.45), 0)

    inv_qty  = round(qty + random.uniform(-2.0, 2.0), 0)
    inv_qty  = max(0.0, inv_qty)
    inv_diff = round(inv_qty - qty, 0)
    inv_date = rand_date()

    lot_ref  = (
        random.choice(lot_id_pool)
        if lot_id_pool and random.random() > 0.6
        else None
    )

    quant_rows.append(Row(
        stock_quant_id          = quant_id,
        product_id              = pid,
        product_name            = prod["name"],
        company_id              = COMPANY_ID,
        company_name            = COMPANY,
        location_id             = lid,
        location_name           = f"LOC-{lid}",
        lot_id                  = lot_ref,
        lot_name                = f"LOT-{now.year}-{lot_ref:04d}" if lot_ref else None,
        quantity                = qty,
        reserved_quantity       = reserved,
        inventory_quantity      = inv_qty,
        inventory_diff_quantity = inv_diff,
        inventory_quantity_set  = True,
        inventory_date          = inv_date,
        in_date                 = rand_date(),
        accounting_date         = inv_date,
        created_at              = now,
        updated_at              = now,
        dwh_loaded_at           = now,
        dwh_source_db           = SOURCE_DB,
    ))
    quant_id += 1

# ═════════════════════════════════════════════════════════════════════════════
# 8. WAREHOUSE ORDERPOINTS  (1 per warehouse per run)
# ═════════════════════════════════════════════════════════════════════════════
op_rows = []
op_id   = op_id_start

for wh in all_wh_ids:
    wh_locs_local = wh_to_locs.get(wh, all_loc_ids) or all_loc_ids
    lid      = random.choice(wh_locs_local)
    pid      = random.choice(pid_list)
    prod     = products_map[pid]
    route_id = random.choice(list(ROUTES.keys()))
    min_qty  = round(random.uniform(5.0, 50.0), 2)
    max_qty  = round(min_qty + random.uniform(50.0, 200.0), 2)
    to_order = round(max(0.0, min_qty - random.uniform(0, min_qty)), 2)

    op_rows.append(Row(
        stock_warehouse_orderpoint_id = op_id,
        warehouse_id                  = wh,
        warehouse_name                = wh_name_map.get(wh, f"WH-{wh}"),
        location_id                   = lid,
        location_name                 = f"LOC-{lid}",
        product_id                    = pid,
        product_name                  = prod["name"],   # ← real name
        company_id                    = COMPANY_ID,
        company_name                  = COMPANY,
        replenishment_uom_id          = 1,
        replenishment_uom_name        = "Unit(s)",
        route_id                      = route_id,
        route_name                    = ROUTES[route_id],
        name                          = f"OP-{wh}-{op_id:04d}",
        trigger                       = random.choice(["orderpoint", "manual"]),
        product_min_qty               = min_qty,
        product_max_qty               = max_qty,
        qty_to_order_computed         = to_order,
        qty_to_order_manual           = 0.0,
        active                        = True,
        snoozed_until                 = maybe_date(0.8),
        deadline_date                 = maybe_date(0.5),
        created_at                    = now,
        updated_at                    = now,
        dwh_loaded_at                 = now,
        dwh_source_db                 = SOURCE_DB,
    ))
    op_id += 1

# ═════════════════════════════════════════════════════════════════════════════
# 9. SCRAP  (20 per run — linked to real products)
# ═════════════════════════════════════════════════════════════════════════════
scrap_rows  = []
scrap_id    = scrap_id_start
SCRAP_STATES = (["draft"] * 30) + (["done"] * 70)

for _ in range(20):
    pid      = random.choice(pid_list)
    prod     = products_map[pid]
    state    = random.choice(SCRAP_STATES)
    loc_s    = random.choice(all_loc_ids) if all_loc_ids else 1
    lot_ref  = random.choice(lot_id_pool) if lot_id_pool and random.random() > 0.5 else None
    pick_ref = random.choice(new_pick_ids) if new_pick_ids and random.random() > 0.6 else None

    scrap_rows.append(Row(
        stock_scrap_id      = scrap_id,
        company_id          = COMPANY_ID,
        company_name        = COMPANY,
        product_id          = pid,
        product_name        = prod["name"],    # ← real name
        product_uom_id      = 1,
        product_uom_name    = "Unit(s)",
        lot_id              = lot_ref,
        lot_name            = f"LOT-{now.year}-{lot_ref:04d}" if lot_ref else None,
        location_id         = loc_s,
        location_name       = f"LOC-{loc_s}",
        scrap_location_id   = random.choice(all_loc_ids) if all_loc_ids else 1,
        scrap_location_name = "Virtual Locations/Scrap",
        picking_id          = pick_ref,
        picking_name        = f"PICK-{pick_ref:06d}" if pick_ref else None,
        name                = f"SCRAP-{scrap_id:05d}",
        origin              = f"SO-{random.randint(1000,9999)}" if random.random() > 0.5 else None,
        state               = state,
        scrap_qty           = round(random.uniform(1.0, 20.0), 2),
        should_replenish    = random.random() < 0.5,
        date_done           = rand_date() if state == "done" else None,
        created_at          = now,
        updated_at          = now,
        dwh_loaded_at       = now,
        dwh_source_db       = SOURCE_DB,
    ))
    scrap_id += 1

# ═════════════════════════════════════════════════════════════════════════════
# SCHEMAS
# ═════════════════════════════════════════════════════════════════════════════
wh_schema = StructType([
    StructField("warehouse_id",    IntegerType(),   True),
    StructField("company_id",      IntegerType(),   True),
    StructField("company_name",    StringType(),    True),
    StructField("partner_id",      IntegerType(),   True),
    StructField("partner_name",    StringType(),    True),
    StructField("warehouse_name",  StringType(),    True),
    StructField("code",            StringType(),    True),
    StructField("reception_steps", StringType(),    True),
    StructField("delivery_steps",  StringType(),    True),
    StructField("active",          BooleanType(),   True),
    StructField("created_at",      TimestampType(), True),
    StructField("updated_at",      TimestampType(), True),
    StructField("dwh_loaded_at",   TimestampType(), True),
    StructField("dwh_source_db",   StringType(),    True),
])

loc_schema = StructType([
    StructField("stock_location_id",   IntegerType(),   True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("location_id",         IntegerType(),   True),
    StructField("location_name",       StringType(),    True),
    StructField("warehouse_id",        IntegerType(),   True),
    StructField("stock_location_name", StringType(),    True),
    StructField("complete_name",       StringType(),    True),
    StructField("usage",               StringType(),    True),
    StructField("parent_path",         StringType(),    True),
    StructField("barcode",             StringType(),    True),
    StructField("active",              BooleanType(),   True),
    StructField("replenish_location",  BooleanType(),   True),
    StructField("last_inventory_date", TimestampType(), True),
    StructField("next_inventory_date", TimestampType(), True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

lot_schema = StructType([
    StructField("stock_lot_id",    IntegerType(),   True),
    StructField("product_id",      IntegerType(),   True),
    StructField("product_name",    StringType(),    True),
    StructField("company_id",      IntegerType(),   True),
    StructField("company_name",    StringType(),    True),
    StructField("location_id",     IntegerType(),   True),
    StructField("location_name",   StringType(),    True),
    StructField("lot_name",        StringType(),    True),
    StructField("ref",             StringType(),    True),
    StructField("note",            StringType(),    True),
    StructField("lot_properties",  StringType(),    True),
    StructField("standard_price",  StringType(),    True),
    StructField("avg_cost",        DoubleType(),    True),
    StructField("created_at",      TimestampType(), True),
    StructField("updated_at",      TimestampType(), True),
    StructField("dwh_loaded_at",   TimestampType(), True),
    StructField("dwh_source_db",   StringType(),    True),
])

pick_schema = StructType([
    StructField("stock_picking_id",   IntegerType(),   True),
    StructField("company_id",         IntegerType(),   True),
    StructField("company_name",       StringType(),    True),
    StructField("partner_id",         IntegerType(),   True),
    StructField("partner_name",       StringType(),    True),
    StructField("picking_type_id",    IntegerType(),   True),
    StructField("picking_type_name",  StringType(),    True),
    StructField("location_id",        IntegerType(),   True),
    StructField("location_name",      StringType(),    True),
    StructField("location_dest_id",   IntegerType(),   True),
    StructField("location_dest_name", StringType(),    True),
    StructField("user_id",            IntegerType(),   True),
    StructField("user_name",          StringType(),    True),
    StructField("backorder_id",       IntegerType(),   True),
    StructField("return_id",          StringType(),    True),
    StructField("stock_picking_name", StringType(),    True),
    StructField("origin",             StringType(),    True),
    StructField("move_type",          StringType(),    True),
    StructField("state",              StringType(),    True),
    StructField("priority",           StringType(),    True),
    StructField("note",               StringType(),    True),
    StructField("shipping_weight",    DoubleType(),    True),
    StructField("has_deadline_issue", BooleanType(),   True),
    StructField("printed",            BooleanType(),   True),
    StructField("is_locked",          BooleanType(),   True),
    StructField("scheduled_date",     TimestampType(), True),
    StructField("date_deadline",      TimestampType(), True),
    StructField("date_done",          TimestampType(), True),
    StructField("created_at",         TimestampType(), True),
    StructField("updated_at",         TimestampType(), True),
    StructField("dwh_loaded_at",      TimestampType(), True),
    StructField("dwh_source_db",      StringType(),    True),
])

move_schema = StructType([
    StructField("stock_move_id",              IntegerType(),   True),
    StructField("company_id",                 IntegerType(),   True),
    StructField("company_name",               StringType(),    True),
    StructField("product_id",                 IntegerType(),   True),
    StructField("product_name",               StringType(),    True),
    StructField("product_uom_id",             IntegerType(),   True),
    StructField("product_uom_name",           StringType(),    True),
    StructField("location_id",                IntegerType(),   True),
    StructField("location_name",              StringType(),    True),
    StructField("location_dest_id",           IntegerType(),   True),
    StructField("location_dest_name",         StringType(),    True),
    StructField("location_final_id",          IntegerType(),   True),
    StructField("location_final_name",        StringType(),    True),
    StructField("partner_id",                 IntegerType(),   True),
    StructField("partner_name",               StringType(),    True),
    StructField("picking_id",                 IntegerType(),   True),
    StructField("picking_name",               StringType(),    True),
    StructField("picking_type_id",            IntegerType(),   True),
    StructField("picking_type_name",          StringType(),    True),
    StructField("warehouse_id",               IntegerType(),   True),
    StructField("warehouse_name",             StringType(),    True),
    StructField("origin_returned_move_id",    IntegerType(),   True),
    StructField("origin_returned_move_name",  StringType(),    True),
    StructField("priority",                   StringType(),    True),
    StructField("state",                      StringType(),    True),
    StructField("origin",                     StringType(),    True),
    StructField("reference",                  StringType(),    True),
    StructField("procure_method",             StringType(),    True),
    StructField("inventory_name",             StringType(),    True),
    StructField("next_serial",                StringType(),    True),
    StructField("next_serial_count",          IntegerType(),   True),
    StructField("product_qty",                DoubleType(),    True),
    StructField("product_uom_qty",            DoubleType(),    True),
    StructField("quantity",                   DoubleType(),    True),
    StructField("price_unit",                 DoubleType(),    True),
    StructField("value",                      DoubleType(),    True),
    StructField("picked",                     BooleanType(),   True),
    StructField("propagate_cancel",           BooleanType(),   True),
    StructField("is_inventory",               BooleanType(),   True),
    StructField("additional",                 BooleanType(),   True),
    StructField("to_refund",                  BooleanType(),   True),
    StructField("is_in",                      BooleanType(),   True),
    StructField("is_out",                     BooleanType(),   True),
    StructField("is_dropship",                BooleanType(),   True),
    StructField("date",                       TimestampType(), True),
    StructField("date_deadline",              TimestampType(), True),
    StructField("reservation_date",           TimestampType(), True),
    StructField("created_at",                 TimestampType(), True),
    StructField("updated_at",                 TimestampType(), True),
    StructField("dwh_loaded_at",              TimestampType(), True),
    StructField("dwh_source_db",              StringType(),    True),
])

ml_schema = StructType([
    StructField("stock_move_line_id",   IntegerType(),   True),
    StructField("move_id",              IntegerType(),   True),
    StructField("picking_id",           StringType(),    True),
    StructField("company_id",           IntegerType(),   True),
    StructField("company_name",         StringType(),    True),
    StructField("product_id",           IntegerType(),   True),
    StructField("product_name",         StringType(),    True),
    StructField("product_uom_id",       IntegerType(),   True),
    StructField("product_uom_name",     StringType(),    True),
    StructField("location_id",          IntegerType(),   True),
    StructField("location_name",        StringType(),    True),
    StructField("location_dest_id",     IntegerType(),   True),
    StructField("location_dest_name",   StringType(),    True),
    StructField("lot_id",               IntegerType(),   True),
    StructField("lot_name",             StringType(),    True),
    StructField("state",                StringType(),    True),
    StructField("quantity",             DoubleType(),    True),
    StructField("quantity_product_uom", DoubleType(),    True),
    StructField("picked",               BooleanType(),   True),
    StructField("is_entire_pack",       BooleanType(),   True),
    StructField("date",                 TimestampType(), True),
    StructField("created_at",           TimestampType(), True),
    StructField("updated_at",           TimestampType(), True),
    StructField("dwh_loaded_at",        TimestampType(), True),
    StructField("dwh_source_db",        StringType(),    True),
])

quant_schema = StructType([
    StructField("stock_quant_id",           IntegerType(),   True),
    StructField("product_id",               IntegerType(),   True),
    StructField("product_name",             StringType(),    True),
    StructField("company_id",               IntegerType(),   True),
    StructField("company_name",             StringType(),    True),
    StructField("location_id",              IntegerType(),   True),
    StructField("location_name",            StringType(),    True),
    StructField("lot_id",                   IntegerType(),   True),
    StructField("lot_name",                 StringType(),    True),
    StructField("quantity",                 DoubleType(),    True),
    StructField("reserved_quantity",        DoubleType(),    True),
    StructField("inventory_quantity",       DoubleType(),    True),
    StructField("inventory_diff_quantity",  DoubleType(),    True),
    StructField("inventory_quantity_set",   BooleanType(),   True),
    StructField("inventory_date",           TimestampType(), True),
    StructField("in_date",                  TimestampType(), True),
    StructField("accounting_date",          TimestampType(), True),
    StructField("created_at",               TimestampType(), True),
    StructField("updated_at",               TimestampType(), True),
    StructField("dwh_loaded_at",            TimestampType(), True),
    StructField("dwh_source_db",            StringType(),    True),
])

op_schema = StructType([
    StructField("stock_warehouse_orderpoint_id", IntegerType(),   True),
    StructField("warehouse_id",                  IntegerType(),   True),
    StructField("warehouse_name",                StringType(),    True),
    StructField("location_id",                   IntegerType(),   True),
    StructField("location_name",                 StringType(),    True),
    StructField("product_id",                    IntegerType(),   True),
    StructField("product_name",                  StringType(),    True),
    StructField("company_id",                    IntegerType(),   True),
    StructField("company_name",                  StringType(),    True),
    StructField("replenishment_uom_id",          IntegerType(),   True),
    StructField("replenishment_uom_name",        StringType(),    True),
    StructField("route_id",                      IntegerType(),   True),
    StructField("route_name",                    StringType(),    True),
    StructField("name",                          StringType(),    True),
    StructField("trigger",                       StringType(),    True),
    StructField("product_min_qty",               DoubleType(),    True),
    StructField("product_max_qty",               DoubleType(),    True),
    StructField("qty_to_order_computed",         DoubleType(),    True),
    StructField("qty_to_order_manual",           DoubleType(),    True),
    StructField("active",                        BooleanType(),   True),
    StructField("snoozed_until",                 TimestampType(), True),
    StructField("deadline_date",                 TimestampType(), True),
    StructField("created_at",                    TimestampType(), True),
    StructField("updated_at",                    TimestampType(), True),
    StructField("dwh_loaded_at",                 TimestampType(), True),
    StructField("dwh_source_db",                 StringType(),    True),
])

scrap_schema = StructType([
    StructField("stock_scrap_id",      IntegerType(),   True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("product_id",          IntegerType(),   True),
    StructField("product_name",        StringType(),    True),
    StructField("product_uom_id",      IntegerType(),   True),
    StructField("product_uom_name",    StringType(),    True),
    StructField("lot_id",              IntegerType(),   True),
    StructField("lot_name",            StringType(),    True),
    StructField("location_id",         IntegerType(),   True),
    StructField("location_name",       StringType(),    True),
    StructField("scrap_location_id",   IntegerType(),   True),
    StructField("scrap_location_name", StringType(),    True),
    StructField("picking_id",          IntegerType(),   True),
    StructField("picking_name",        StringType(),    True),
    StructField("name",                StringType(),    True),
    StructField("origin",              StringType(),    True),
    StructField("state",               StringType(),    True),
    StructField("scrap_qty",           DoubleType(),    True),
    StructField("should_replenish",    BooleanType(),   True),
    StructField("date_done",           TimestampType(), True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

# ═════════════════════════════════════════════════════════════════════════════
# CREATE DATAFRAMES
# ═════════════════════════════════════════════════════════════════════════════
df_wh    = spark.createDataFrame(wh_rows,    schema=wh_schema)
df_loc   = spark.createDataFrame(loc_rows,   schema=loc_schema)
df_lot   = spark.createDataFrame(lot_rows,   schema=lot_schema)
df_pick  = spark.createDataFrame(pick_rows,  schema=pick_schema)
df_move  = spark.createDataFrame(move_rows,  schema=move_schema)
df_ml    = spark.createDataFrame(ml_rows,    schema=ml_schema)
df_quant = spark.createDataFrame(quant_rows, schema=quant_schema)
df_op    = spark.createDataFrame(op_rows,    schema=op_schema)
df_scrap = spark.createDataFrame(scrap_rows, schema=scrap_schema)

print(f"\n  stock_warehouse rows             : {df_wh.count()}")
print(f"  stock_location rows              : {df_loc.count()}")
print(f"  stock_lot rows                   : {df_lot.count()}")
print(f"  stock_picking rows               : {df_pick.count()}")
print(f"  stock_move rows                  : {df_move.count()}")
print(f"  stock_move_line rows             : {df_ml.count()}")
print(f"  stock_quant rows                 : {df_quant.count()}")
print(f"  stock_warehouse_orderpoint rows  : {df_op.count()}")
print(f"  stock_scrap rows                 : {df_scrap.count()}")

# ═════════════════════════════════════════════════════════════════════════════
# WRITE TO DELTA
# ═════════════════════════════════════════════════════════════════════════════
def append_delta(df, table: str):
    (
        df.write
          .format("delta")
          .mode("append")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
    )
    print(f"  ✅  {table}")

print("\nWriting to Delta …")
append_delta(df_wh,    "stock_warehouse")
append_delta(df_loc,   "stock_location")
append_delta(df_lot,   "stock_lot")
append_delta(df_pick,  "stock_picking")
append_delta(df_move,  "stock_move")
append_delta(df_ml,    "stock_move_line")
append_delta(df_quant, "stock_quant")
append_delta(df_op,    "stock_warehouse_orderpoint")
append_delta(df_scrap, "stock_scrap")

print(f"""
✅  Stock simulation complete.
    Pickings  : {len(pick_rows)}  (receipts=150 | deliveries=120 | internal=30 | returns=10)
    Moves     : {len(move_rows)}
    Move lines: {len(ml_rows)}
    Quants    : {len(quant_rows)}  (all qty > 0, reserved ≤ on-hand)
    Lots      : {len(lot_rows)}   (linked to real products)
    Scrap     : {len(scrap_rows)}
""")